# Module 04: PUF & TRNG — Lab

This lab simulates PUF behavior and TRNG entropy assessment.

**Objectives:**
1. Simulate RO-PUF behavior with manufacturing variations
2. Analyze PUF reliability and uniqueness metrics
3. Generate and test TRNG-like random sequences
4. Perform NIST-style statistical tests on random data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("=" * 60)
print("RO-PUF SIMULATION")
print("=" * 60)

# Simulate RO-PUF: manufacturing variations in oscillator frequencies
np.random.seed(42)
n_chips = 50    # Number of different chips
n_ros = 128     # Number of ring oscillators per chip

# Each chip has unique manufacturing variation (die-specific)
# Each RO within a chip also has variation
chip_bias = np.random.normal(0, 0.01, n_chips)  # Per-chip process variation
ro_variations = np.random.normal(0, 0.005, (n_chips, n_ros))  # Per-RO variation

# Base frequency (normalized)
base_freq = 1.0
frequencies = np.zeros((n_chips, n_ros))
for i in range(n_chips):
    frequencies[i] = base_freq + chip_bias[i] + ro_variations[i]

# Generate PUF responses by comparing RO pairs
n_pairs = n_ros // 2
responses = np.zeros((n_chips, n_pairs), dtype=int)
for i in range(n_chips):
    for j in range(n_pairs):
        responses[i, j] = 1 if frequencies[i, 2*j] > frequencies[i, 2*j+1] else 0

print(f"Chips simulated: {n_chips}")
print(f"Ring oscillators per chip: {n_ros}")
print(f"PUF response bits per chip: {n_pairs}")
print(f"\nSample response (chip 0): {responses[0, :16]}...")
print(f"Sample response (chip 1): {responses[1, :16]}...")

In [ ]:
# PUF Metrics Calculation
def hamming_distance(a, b):
    return np.sum(a != b)

# Uniqueness: average inter-chip Hamming distance
hd_values = []
for i in range(n_chips):
    for j in range(i+1, n_chips):
        hd = hamming_distance(responses[i], responses[j])
        hd_values.append(hd / n_pairs)

uniqueness = np.mean(hd_values)
print(f"PUF UNIQUENESS (inter-chip HD): {uniqueness:.4f} (ideal: 0.5)")

# Uniformity: average fraction of 1s per chip
uniformity_values = [np.mean(responses[i]) for i in range(n_chips)]
uniformity = np.mean(uniformity_values)
print(f"PUF UNIFORMITY (avg fraction of 1s): {uniformity:.4f} (ideal: 0.5)")

# Reliability: simulate noisy re-measurement
noise_rate = 0.03  # 3% bit error rate
noisy_responses = np.zeros_like(responses)
for i in range(n_chips):
    noise_mask = np.random.random(n_pairs) < noise_rate
    noisy_responses[i] = responses[i] ^ noise_mask.astype(int)

reliability_values = [1 - hamming_distance(responses[i], noisy_responses[i])/n_pairs for i in range(n_chips)]
reliability = np.mean(reliability_values)
print(f"PUF RELIABILITY: {reliability:.4f} (ideal: 1.0)")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(hd_values, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(0.5, color='r', linestyle='--', label='Ideal (0.5)')
axes[0].set_title('Inter-chip Hamming Distance')
axes[0].set_xlabel('Normalized HD')
axes[0].legend()

axes[1].hist(uniformity_values, bins=20, edgecolor='black', alpha=0.7)
axes[1].axvline(0.5, color='r', linestyle='--', label='Ideal (0.5)')
axes[1].set_title('Per-chip Uniformity')
axes[1].set_xlabel('Fraction of 1s')
axes[1].legend()

axes[2].bar(range(n_chips), reliability_values, alpha=0.7)
axes[2].axhline(1.0, color='r', linestyle='--', label='Ideal (1.0)')
axes[2].set_title('Per-chip Reliability')
axes[2].set_xlabel('Chip Index')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# TRNG Simulation and Statistical Testing
print("=" * 60)
print("TRNG SIMULATION & NIST-STYLE TESTING")
print("=" * 60)

# Simulate TRNG output using ring oscillator jitter model
n_bits = 10000

# Good TRNG (uniform random)
good_trng = np.random.randint(0, 2, n_bits)

# Biased TRNG (slightly biased)
biased_trng = (np.random.random(n_bits) < 0.55).astype(int)  # 55% chance of 1

# Periodic TRNG (defective)
periodic_trng = np.tile([0, 1, 1, 0], n_bits // 4)

def frequency_test(bits):
    """Monobit frequency test (NIST SP 800-22 Test 1)"""
    n = len(bits)
    s = 2 * bits.sum() - n  # Convert 0/1 to -1/+1
    s_obs = abs(s) / np.sqrt(n)
    from scipy.stats import erfc
    p_value = erfc(s_obs / np.sqrt(2))
    return p_value

def runs_test(bits):
    """Runs test (NIST SP 800-22 Test 2)"""
    n = len(bits)
    pi = bits.mean()
    if abs(pi - 0.5) >= 2/np.sqrt(n):
        return 0.0  # Non-random
    
    # Count runs
    runs = 1
    for i in range(1, n):
        if bits[i] != bits[i-1]:
            runs += 1
    
    expected_runs = 1 + 2 * n * pi * (1 - pi)
    var_runs = 2 * n * pi * (1 - pi) * (2 * n * pi * (1 - pi) - 1) / (n - 1)
    z = abs(runs - expected_runs) / np.sqrt(var_runs)
    from scipy.stats import erfc
    return erfc(z / np.sqrt(2))

# Test all three TRNG outputs
for name, trng in [("Good TRNG", good_trng), ("Biased TRNG", biased_trng), ("Periodic TRNG", periodic_trng)]:
    try:
        freq_p = frequency_test(trng)
        runs_p = runs_test(trng)
    except:
        freq_p = 'N/A (scipy needed)'
        runs_p = 'N/A (scipy needed)'
    
    print(f"\n{name}:")
    print(f"  Fraction of 1s: {trng.mean():.4f}")
    print(f"  Frequency test p-value: {freq_p}")
    print(f"  Runs test p-value: {runs_p}")
    if isinstance(freq_p, float):
        print(f"  Frequency test: {'PASS' if freq_p > 0.01 else 'FAIL'} (alpha=0.01)")

# Entropy estimation
print("\n" + "=" * 60)
print("ENTROPY ESTIMATION")
print("=" * 60)

def min_entropy(bits):
    p_max = max(bits.mean(), 1 - bits.mean())
    return -np.log2(p_max)

def shannon_entropy(bits):
    p = bits.mean()
    if p == 0 or p == 1:
        return 0
    return -(p * np.log2(p) + (1-p) * np.log2(1-p))

for name, trng in [("Good TRNG", good_trng), ("Biased TRNG", biased_trng)]:
    h_min = min_entropy(trng)
    h_shannon = shannon_entropy(trng)
    print(f"{name}: Min-entropy = {h_min:.4f} bits/bit, Shannon entropy = {h_shannon:.4f} bits/bit")

print("\nMin-entropy is the conservative measure used by NIST SP 800-90B.")
print("A good TRNG should have min-entropy close to 1.0 bits/bit.")